## Notebook 02: Hydrogen Case Study Project Setup

This notebook creates the foreground databases for the hydrogen production case study. It imports hydrogen and methanol inventories from Excel, links them to background databases, and creates regionalized (country-specific) versions of each foreground database.

**What it does:**
1. Imports methanol and hydrogen foreground inventories from Excel into Brightway.
2. Links foreground exchanges to ecoinvent and Regioinvent background databases.
3. Creates regionalized foreground databases by cloning activities per country using semantic matching to resolve location-specific exchanges.
4. Optionally creates a regionalized ecoinvent subset database for specific activities.
5. Adds negative biogenic CO2 flows where applicable.

**Inputs:**
- Brightway project from notebooks 00 and 01.
- Hydrogen inventory Excel (`Data/Hydrogen_Inventories/Hydrogen.xlsx`).
- Methanol inventory Excel (`Data/Hydrogen_Inventories/Methanol.xlsx`).

**Outputs:**
- Foreground databases in Brightway: `01_hydrogen_inventories_regionalized`, `02_methanol_inventories_regionalized`, and an ecoinvent subset database.

## Import Dependencies

In [ ]:
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
from typing import Dict, Iterable, List, Optional, Sequence, Tuple, Any

import pandas as pd
from tqdm.auto import tqdm

import bw2data as bd
from bw2io import ExcelImporter

from sentence_transformers import SentenceTransformer, util

import numpy as np
np.NaN = np.nan

## Configuration

### 1. File Paths

In [ ]:
# Foreground Excel files
METHANOL_EXCEL = Path(r"../Data/Hydrogen_Inventories/Methanol.xlsx")
HYDROGEN_EXCEL = Path(r"../Data/Hydrogen_Inventories/Hydrogen.xlsx")

# Optional troubleshooting exports for unlinked exchanges
UNLINKED_METHANOL_CSV = Path(r"../Data/Hydrogen_Inventories/unlinked_exchanges_methanol.csv")
UNLINKED_HYDROGEN_CSV = Path(r"../Data/Hydrogen_Inventories/unlinked_exchanges_hydrogen.csv")

# Log files
CO2_PATCH_LOG_XLSX = Path("added_negative_co2_flows.xlsx")

### 2. Project and Database Parameters

In [ ]:
# Brightway project
PROJECT = "sense_project"

# Core background databases
BIOSPHERE_DB = "biosphere3_spatialized_flows"
ECOINVENT_DB = "ecoinvent-3.10.1-cutoff regionalized"
SOCIOSPHERE_DB = "sociosphere"  # Social flows stored separately

# Foreground database names (as written by ExcelImporter)
# IMPORTANT: ExcelImporter writes the database name based on the Excel content.
# If your Excel contains a different DB name, set it below accordingly.
METHANOL_FG_DB_NAME = "02_methanol_inventories"
HYDROGEN_FG_DB_NAME = "01_hydrogen_inventories"

# Localized database names
METHANOL_LOCALIZED_DB = "02_methanol_inventories_regionalized"
HYDROGEN_LOCALIZED_DB = "01_hydrogen_inventories_regionalized"

# Localized ecoinvent subset database
ECOINVENT_SUBSET_OUTPUT_DB = "01_hydrogen_inventories_regionalized_subset"
ECOINVENT_SUBSET_SOURCE_DB = ECOINVENT_DB

### 3. Localization Setup

In [ ]:
# Semantic matching configuration
MODEL_NAME = "all-MiniLM-L6-v2"
SIMILARITY_THRESHOLD = 0.75

# Methanol localization
METHANOL_BG_DBS_FOR_LOCALIZATION = [ECOINVENT_DB, "Regioinvent"]
METHANOL_EXTRACT_LOCATIONS_FROM_BG_INDEX = 1  # index in list above; 1 means "Regioinvent" here

# Hydrogen localization
HYDROGEN_BG_DBS_FOR_LOCALIZATION = [
    METHANOL_LOCALIZED_DB,
    ECOINVENT_DB,
    "Regioinvent",
]
HYDROGEN_EXTRACT_LOCATIONS_FROM_BG_INDEX = 2  # index of BG DB used to extract locations (Regioinvent)

# Where hydrogen FG should match additional foreground DB(s) during import (for biomethane market link)
HYDROGEN_MATCH_EXTRA_FG_DB = METHANOL_FG_DB_NAME

# Ecoinvent subset localization
ECOINVENT_SUBSET_BG_DBS = [ECOINVENT_DB, "Regioinvent"]
ECOINVENT_SUBSET_EXTRACT_LOCATIONS_FROM_BG_INDEX = 1

### 4. Optional Features

In [ ]:
# Overwrite behavior
OVERWRITE_DATABASES = True

# Build localized ecoinvent subset database
BUILD_LOCALIZED_ECOINVENT_SUBSET = True

# Target activities to clone/localize from ecoinvent (exact tuples)
ECOINVENT_SUBSET_TARGET_ACTS = {
    ("hydrogen production, coal gasification", "hydrogen, gaseous, low pressure", "CN"),
    ("hydrogen production, gaseous, petroleum refinery operation", "hydrogen, gaseous, low pressure", "IN"),
    ("hydrogen production, steam methane reforming", "hydrogen, gaseous, low pressure", "CN"),
}

# Social biosphere flow removal config
SOCIAL_COMPARTMENTS = {
    "Children",
    "Consumer",
    "Local Community",
    "Society",
    "Workers",
    "Value Chain Actors",
}

# CO2 method patch config
PATCH_NEGATIVE_CO2 = True
CO2_FLOW_NAME = "Carbon dioxide, in air"
CO2_FLOW_CATEGORIES = ("natural resource", "in air")
CO2_KEYWORD = "carbon dioxide"

## Helper Functions

### Project and Database Utilities

In [ ]:
def _ensure_project(project: str) -> None:
    """Set the current Brightway project."""
    bd.projects.set_current(project)


def _maybe_overwrite_db(db_name: str, overwrite: bool) -> None:
    """Delete database if it exists and overwrite is enabled."""
    if db_name in bd.databases:
        if overwrite:
            print(f"⚠️ Overwriting existing DB: {db_name}")
            del bd.databases[db_name]
        else:
            raise ValueError(f"Database already exists and overwrite=False: {db_name}")

### Foreground Database Import and Linking

In [ ]:
def import_foreground_from_excel(
    excel_path: Path,
    biosphere_db: str,
    technosphere_db: str,
    extra_match_dbs: Optional[Sequence[str]] = None,
    unlinked_csv: Optional[Path] = None,
) -> None:
    """
    Import a BW2 foreground database from an Excel file and link exchanges.

    Linking order:
      1) match to itself (foreground)
      2) match to technosphere background DB
      3) match to any additional DBs (e.g., another foreground DB)
      4) match to biosphere DB

    Parameters
    ----------
    excel_path : Path
        Path to the Excel file containing foreground inventory data
    biosphere_db : str
        Name of the biosphere database for linking biosphere flows
    technosphere_db : str
        Name of the main technosphere background database
    extra_match_dbs : Optional[Sequence[str]]
        Additional databases to match against (e.g., other foreground DBs)
    unlinked_csv : Optional[Path]
        If provided, writes unlinked exchanges to this CSV file
    """
    excel_path = Path(excel_path)
    if not excel_path.exists():
        raise FileNotFoundError(f"Excel not found: {excel_path}")

    imp = ExcelImporter(str(excel_path))
    imp.apply_strategies()

    # Foreground-to-foreground
    imp.match_database(fields=["name", "unit", "reference product", "location"])

    # Background technosphere
    imp.match_database(technosphere_db, fields=["name", "unit", "location", "reference product"])

    # Extra DB(s) (e.g., another foreground DB)
    if extra_match_dbs:
        for dbn in extra_match_dbs:
            imp.match_database(dbn, fields=["name", "unit", "location", "reference product"])

    # Biosphere
    imp.match_database(biosphere_db, fields=["name", "categories", "location"])

    imp.statistics()

    # Export unlinked
    if unlinked_csv is not None:
        unlinked_list = list(imp.unlinked)
        if unlinked_list:
            pd.DataFrame(unlinked_list).to_csv(unlinked_csv, index=False)
            print(f"⚠️ Unlinked exchanges written to: {unlinked_csv}")
        else:
            print("✅ All exchanges successfully linked. No unlinked exchanges to export.")

    # Write DB
    imp.write_database()
    print("✅ Foreground database installed successfully.")

### Database Post-Processing

In [ ]:
def remove_self_referencing_technosphere_exchanges(db_name: str) -> int:
    """
    Remove technosphere exchanges where an activity consumes itself (exc.input == act.key).
    
    Parameters
    ----------
    db_name : str
        Name of the database to process
        
    Returns
    -------
    int
        Number of removed exchanges
    """
    db = bd.Database(db_name)
    removed = 0
    for act in db:
        modified = False
        for exc in list(act.technosphere()):
            if exc.input == act.key:
                exc.delete()
                removed += 1
                modified = True
        if modified:
            act.save()
    return removed


def replace_technosphere_exchange_in_activity(
    db_name: str,
    target_activity_name: str,
    target_reference_product: str,
    old_input_names: Sequence[str],
    old_input_reference_product: str,
    new_input_name: str,
    new_input_reference_product: str,
) -> Tuple[int, int]:
    """
    For each location where the target activity exists:
      - Find the replacement provider (new_input_name/new_input_reference_product) in same location.
      - Replace any matching old input exchange (by input activity name + reference product),
        preserving the exchange amount.

    Parameters
    ----------
    db_name : str
        Name of the database containing the target activity
    target_activity_name : str
        Name of the activity to modify
    target_reference_product : str
        Reference product of the target activity
    old_input_names : Sequence[str]
        List of old input activity names to replace
    old_input_reference_product : str
        Reference product of the old input
    new_input_name : str
        Name of the new input activity
    new_input_reference_product : str
        Reference product of the new input
        
    Returns
    -------
    Tuple[int, int]
        (replacements, skipped) - number of successful replacements and skipped locations
    """
    db = bd.Database(db_name)

    def find_activity(name: str, ref_product: str, location: str):
        hits = [
            act for act in db
            if act.get("name") == name
            and act.get("reference product") == ref_product
            and act.get("location") == location
        ]
        if len(hits) == 1:
            return hits[0]
        if len(hits) == 0:
            return None
        raise ValueError(f"Multiple matches for ({name}, {ref_product}, {location}) in {db_name}")

    replacements = 0
    skipped = 0

    for act in db:
        if act.get("name") != target_activity_name:
            continue
        if act.get("reference product") != target_reference_product:
            continue

        loc = act.get("location")
        new_act = find_activity(new_input_name, new_input_reference_product, loc)
        if not new_act:
            print(f"⏭️ Skipping {loc} — new exchange not found")
            skipped += 1
            continue

        modified = False
        for exc in list(act.technosphere()):
            input_act = bd.get_activity(exc.input)
            if input_act.get("name") in set(old_input_names) and input_act.get("reference product") == old_input_reference_product:
                amount = exc["amount"]
                exc.delete()
                act.new_exchange(input=new_act.key, amount=amount, type="technosphere").save()
                modified = True

        if modified:
            act.save()
            replacements += 1
        else:
            print(f"⏭️ Skipping {loc} — matching old exchange not found in technosphere")
            skipped += 1

    return replacements, skipped

### Location Extraction and Semantic Matching Utilities

In [ ]:
def extract_target_locations_from_db(
    db_name: str,
    exclude: Sequence[str] = ("GLO", "RoW"),
) -> List[str]:
    """
    Extract unique locations from a BW database, excluding common aggregates.
    
    Parameters
    ----------
    db_name : str
        Name of the database to extract locations from
    exclude : Sequence[str]
        Location codes to exclude (typically global/rest-of-world)
        
    Returns
    -------
    List[str]
        Sorted list of unique location codes
    """
    db = bd.Database(db_name)
    locs = sorted({act.get("location") for act in db if act.get("location") not in set(exclude)})
    return locs


def build_candidate_index(background_db_names: Sequence[str]) -> Dict[Tuple[str, str, str], List[Any]]:
    """
    Build pooled candidate index across multiple background databases.
    
    Parameters
    ----------
    background_db_names : Sequence[str]
        List of database names to pool
        
    Returns
    -------
    Dict[Tuple[str, str, str], List[Any]]
        Index mapping (reference product, unit, location) to list of activities
        Order matters for determinism: activities are not sorted
    """
    idx: Dict[Tuple[str, str, str], List[Any]] = {}
    for dbn in background_db_names:
        db = bd.Database(dbn)
        for act in db:
            key = (act.get("reference product"), act.get("unit"), act.get("location"))
            idx.setdefault(key, []).append(act)
    return idx


def best_semantic_match(model: SentenceTransformer, reference_name: str, candidate_names: List[str]) -> Tuple[Optional[str], Optional[float]]:
    """
    Find the best semantic match using SentenceTransformer cosine similarity.
    
    Parameters
    ----------
    model : SentenceTransformer
        Pre-loaded sentence transformer model
    reference_name : str
        Name of the activity to match
    candidate_names : List[str]
        List of candidate activity names
        
    Returns
    -------
    Tuple[Optional[str], Optional[float]]
        (best_candidate_name, similarity_score) or (None, None) if no candidates
    """
    if not candidate_names:
        return None, None
    embeddings = model.encode([reference_name] + candidate_names, convert_to_tensor=True)
    scores = util.cos_sim(embeddings[0], embeddings[1:])[0]
    best_index = scores.argmax().item()
    return candidate_names[best_index], float(scores[best_index].item())

### Foreground Database Localization

In [ ]:
def localize_foreground_database(
    original_fg_db_name: str,
    output_fg_db_name: str,
    background_db_names: Sequence[str],
    target_locations: Sequence[str],
    similarity_threshold: float,
    model: SentenceTransformer,
    overwrite: bool,
    mapping_log_xlsx: Path,
) -> None:
    """
    Clone a foreground DB into many locations and localize technosphere exchanges.

    Rules:
    - Create one localized copy of each activity for each target location (code suffix: _{loc})
    - Always redirect FG->FG technosphere exchanges to the localized FG copy for that location (FG-first)
    - If an input is already in the target location, keep it
    - Otherwise look for background candidates with same (ref product, unit, target location) and
      semantic-match on activity name; swap if similarity >= threshold
    - If no candidates / low similarity, keep original exchange

    Parameters
    ----------
    original_fg_db_name : str
        Name of the original foreground database
    output_fg_db_name : str
        Name for the new localized database
    background_db_names : Sequence[str]
        List of background databases to search for matching activities
    target_locations : Sequence[str]
        List of location codes to create localized versions for
    similarity_threshold : float
        Minimum cosine similarity for semantic matching
    model : SentenceTransformer
        Pre-loaded sentence transformer model
    overwrite : bool
        Whether to overwrite existing output database
    mapping_log_xlsx : Path
        Path to save the exchange mapping log
    """
    _maybe_overwrite_db(output_fg_db_name, overwrite)

    original_db = bd.Database(original_fg_db_name)

    # pooled candidates
    candidate_index = build_candidate_index(background_db_names)

    # precompute expected localized keys for FG-first redirection
    expected_localized_keys = {
        (output_fg_db_name, act.key[1] + f"_{loc}")
        for act in original_db
        for loc in target_locations
    }

    log_entries: List[dict] = []
    new_db_data: Dict[Tuple[str, str], dict] = {}

    for loc in target_locations:
        for act in tqdm(list(original_db), desc=f"Localizing '{original_fg_db_name}' for {loc}"):
            localized_key = (output_fg_db_name, act.key[1] + f"_{loc}")

            new_act_data = {
                "name": act["name"],
                "reference product": act.get("reference product"),
                "unit": act.get("unit"),
                "location": loc,
                "exchanges": [{"input": localized_key, "amount": 1.0, "type": "production"}],
            }

            for exc in act.exchanges():
                if exc["type"] == "production":
                    continue

                # Keep non-technosphere exchanges
                if exc["type"] != "technosphere":
                    new_act_data["exchanges"].append(exc.as_dict())
                    continue

                input_act = bd.get_activity(exc.input)

                # FG-first: redirect to localized FG copy if the input is in the same original FG DB
                if input_act.key[0] == original_fg_db_name:
                    localized_fg_key_in = (output_fg_db_name, input_act.key[1] + f"_{loc}")
                    if localized_fg_key_in in expected_localized_keys:
                        new_act_data["exchanges"].append({
                            "input": localized_fg_key_in,
                            "amount": exc["amount"],
                            "type": "technosphere",
                        })
                        log_entries.append({
                            "target_location": loc,
                            "activity": act.get("name"),
                            "product": input_act.get("name"),
                            "original_location": input_act.get("location"),
                            "new_location": loc,
                            "matched_name": input_act.get("name"),
                            "similarity": 1.0,
                            "status": "redirected to localized FG (FG-first)",
                        })
                        continue

                # Already localized to target location
                if input_act.get("location") == loc:
                    new_act_data["exchanges"].append(exc.as_dict())
                    log_entries.append({
                        "target_location": loc,
                        "activity": act.get("name"),
                        "product": input_act.get("name"),
                        "original_location": input_act.get("location"),
                        "new_location": input_act.get("location"),
                        "matched_name": input_act.get("name"),
                        "similarity": 1.0,
                        "status": "unchanged (already localized)",
                    })
                    continue

                # Background candidates by (ref product, unit, target location)
                k = (input_act.get("reference product"), input_act.get("unit"), loc)
                candidates = candidate_index.get(k, [])

                if not candidates:
                    new_act_data["exchanges"].append(exc.as_dict())
                    log_entries.append({
                        "target_location": loc,
                        "activity": act.get("name"),
                        "product": input_act.get("name"),
                        "original_location": input_act.get("location"),
                        "new_location": input_act.get("location"),
                        "matched_name": None,
                        "similarity": None,
                        "status": "unchanged (no BG candidates)",
                    })
                    continue

                candidate_names = [c.get("name") for c in candidates]
                best_name, score = best_semantic_match(model, input_act.get("name"), candidate_names)

                if best_name is None or score is None or score < similarity_threshold:
                    new_act_data["exchanges"].append(exc.as_dict())
                    log_entries.append({
                        "target_location": loc,
                        "activity": act.get("name"),
                        "product": input_act.get("name"),
                        "original_location": input_act.get("location"),
                        "new_location": input_act.get("location"),
                        "matched_name": best_name,
                        "similarity": round(score, 4) if score is not None else None,
                        "status": f"unchanged (low BG similarity: {round(score, 4) if score is not None else 'n/a'})",
                    })
                    continue

                matched = next(c for c in candidates if c.get("name") == best_name)
                new_act_data["exchanges"].append({
                    "input": matched.key,
                    "amount": exc["amount"],
                    "type": "technosphere",
                })
                log_entries.append({
                    "target_location": loc,
                    "activity": act.get("name"),
                    "product": input_act.get("name"),
                    "original_location": input_act.get("location"),
                    "new_location": matched.get("location"),
                    "matched_name": best_name,
                    "similarity": round(score, 4),
                    "status": "changed (BG match)",
                })

            new_db_data[localized_key] = new_act_data

    bd.Database(output_fg_db_name).write(new_db_data)
    bd.Database(output_fg_db_name).process()
    print(f"✅ Localized DB written to: {output_fg_db_name}")

    pd.DataFrame(log_entries).to_excel(mapping_log_xlsx, index=False)
    print(f"📝 Mapping log saved to: {mapping_log_xlsx}")

### Ecoinvent Subset Localization with Social Flow Removal

In [ ]:
def is_target_activity(act: Any, targets: set[tuple]) -> bool:
    """
    Check if activity matches any target specification.
    
    Parameters
    ----------
    act : Any
        Brightway activity object
    targets : set[tuple]
        Set of (name, ref_product, location) tuples for exact matching
        
    Returns
    -------
    bool
        True if activity matches a target
    """
    return (act.get("name"), act.get("reference product"), act.get("location")) in targets


def is_social_biosphere_exchange(exc: Any) -> bool:
    """
    Return True if biosphere exchange belongs to a social stakeholder compartment.

    Matches if ANY category element:
      - equals one of SOCIAL_COMPARTMENTS, or
      - starts with '<base> -' (e.g., 'Workers - rights')
      
    Parameters
    ----------
    exc : Any
        Brightway exchange object
        
    Returns
    -------
    bool
        True if exchange is a social biosphere flow
    """
    if exc.get("type") != "biosphere":
        return False

    flow = bd.get_activity(exc.input)
    cats = flow.get("categories") or ()

    if isinstance(cats, str):
        cats = (cats,)
    elif not isinstance(cats, (list, tuple)):
        cats = (str(cats),)

    for raw_cat in cats:
        cat = str(raw_cat).strip()
        if cat in SOCIAL_COMPARTMENTS:
            return True
        for base in SOCIAL_COMPARTMENTS:
            if cat.startswith(base + " -"):
                return True
    return False


def localize_ecoinvent_subset(
    source_db_name: str,
    output_db_name: str,
    background_db_names: Sequence[str],
    target_locations: Sequence[str],
    targets: set[tuple],
    model: SentenceTransformer,
    similarity_threshold: float,
    overwrite: bool,
    log_xlsx: Path,
) -> None:
    """
    Create a localized DB by cloning only a subset of activities from a source background DB
    (e.g., ecoinvent hydrogen activities). Removes *social* biosphere flows from clones.

    Matching technosphere inputs uses pooled background candidates by (ref product, unit, loc)
    + semantic match on name; if no/low match, keep original (unless redirected to another cloned subset activity).
    
    Parameters
    ----------
    source_db_name : str
        Source database to clone activities from
    output_db_name : str
        Name for the new localized subset database
    background_db_names : Sequence[str]
        List of background databases for matching
    target_locations : Sequence[str]
        Locations to create clones for
    targets : set[tuple]
        Set of (name, ref_product, location) tuples identifying activities to clone
    model : SentenceTransformer
        Pre-loaded sentence transformer model
    similarity_threshold : float
        Minimum similarity for semantic matching
    overwrite : bool
        Whether to overwrite existing output database
    log_xlsx : Path
        Path to save processing log
    """
    _maybe_overwrite_db(output_db_name, overwrite)

    bg_dbs = [bd.Database(dbn) for dbn in background_db_names]
    source_db = bd.Database(source_db_name)

    # select target acts once (deterministic order)
    target_acts = [act for act in source_db if is_target_activity(act, targets)]
    print(f"Found {len(target_acts)} target activities to localize from '{source_db_name}'.")

    # pooled candidates
    candidate_index = build_candidate_index(background_db_names)

    expected_localized_keys = {
        (output_db_name, act.key[1] + f"_{loc}")
        for act in target_acts
        for loc in target_locations
    }

    log_rows: List[dict] = []
    new_db_data: Dict[Tuple[str, str], dict] = {}

    for loc in target_locations:
        for act in tqdm(target_acts, desc=f"Localizing subset for {loc}"):
            localized_key = (output_db_name, act.key[1] + f"_{loc}")

            new_act_data = {
                "name": act.get("name"),
                "reference product": act.get("reference product"),
                "unit": act.get("unit"),
                "location": loc,
                "exchanges": [{"input": localized_key, "amount": 1.0, "type": "production"}],
            }

            for exc in act.exchanges():
                if exc.get("type") == "production":
                    continue

                # Remove only social biosphere flows; keep all other non-technosphere exchanges
                if exc.get("type") != "technosphere":
                    if is_social_biosphere_exchange(exc):
                        flow = bd.get_activity(exc.input)
                        log_rows.append({
                            "target_location": loc,
                            "activity": act.get("name"),
                            "product": flow.get("name"),
                            "original_location": flow.get("location"),
                            "new_location": None,
                            "matched_name": flow.get("name"),
                            "similarity": None,
                            "status": "removed social biosphere flow",
                        })
                        continue
                    new_act_data["exchanges"].append(exc.as_dict())
                    continue

                # technosphere
                input_act = bd.get_activity(exc.input)

                # already in target loc
                if input_act.get("location") == loc:
                    new_act_data["exchanges"].append(exc.as_dict())
                    log_rows.append({
                        "target_location": loc,
                        "activity": act.get("name"),
                        "product": input_act.get("name"),
                        "original_location": input_act.get("location"),
                        "new_location": input_act.get("location"),
                        "matched_name": input_act.get("name"),
                        "similarity": 1.0,
                        "status": "unchanged (already localized)",
                    })
                    continue

                # BG candidates
                k = (input_act.get("reference product"), input_act.get("unit"), loc)
                candidates = candidate_index.get(k, [])

                if not candidates:
                    # subset fallback redirect
                    if input_act.key[0] == source_db_name and is_target_activity(input_act, targets):
                        redirected = (output_db_name, input_act.key[1] + f"_{loc}")
                        if redirected in expected_localized_keys:
                            new_act_data["exchanges"].append({
                                "input": redirected,
                                "amount": exc.get("amount"),
                                "type": "technosphere",
                            })
                            log_rows.append({
                                "target_location": loc,
                                "activity": act.get("name"),
                                "product": input_act.get("name"),
                                "original_location": input_act.get("location"),
                                "new_location": loc,
                                "matched_name": input_act.get("name"),
                                "similarity": 1.0,
                                "status": "redirected to localized subset",
                            })
                            continue

                    # else keep original
                    new_act_data["exchanges"].append(exc.as_dict())
                    log_rows.append({
                        "target_location": loc,
                        "activity": act.get("name"),
                        "product": input_act.get("name"),
                        "original_location": input_act.get("location"),
                        "new_location": input_act.get("location"),
                        "matched_name": None,
                        "similarity": None,
                        "status": "unchanged (no candidates)",
                    })
                    continue

                candidate_names = [c.get("name") for c in candidates]
                best_name, score = best_semantic_match(model, input_act.get("name"), candidate_names)

                if best_name is None or score is None or score < similarity_threshold:
                    # fallback redirect if subset
                    if input_act.key[0] == source_db_name and is_target_activity(input_act, targets):
                        redirected = (output_db_name, input_act.key[1] + f"_{loc}")
                        if redirected in expected_localized_keys:
                            new_act_data["exchanges"].append({
                                "input": redirected,
                                "amount": exc.get("amount"),
                                "type": "technosphere",
                            })
                            log_rows.append({
                                "target_location": loc,
                                "activity": act.get("name"),
                                "product": input_act.get("name"),
                                "original_location": input_act.get("location"),
                                "new_location": loc,
                                "matched_name": input_act.get("name"),
                                "similarity": 1.0,
                                "status": "redirected to localized subset (low similarity fallback)",
                            })
                            continue

                    new_act_data["exchanges"].append(exc.as_dict())
                    log_rows.append({
                        "target_location": loc,
                        "activity": act.get("name"),
                        "product": input_act.get("name"),
                        "original_location": input_act.get("location"),
                        "new_location": input_act.get("location"),
                        "matched_name": best_name,
                        "similarity": round(score, 4) if score is not None else None,
                        "status": f"unchanged (low similarity: {round(score, 4) if score is not None else 'n/a'})",
                    })
                    continue

                matched = next(c for c in candidates if c.get("name") == best_name)
                new_act_data["exchanges"].append({
                    "input": matched.key,
                    "amount": exc.get("amount"),
                    "type": "technosphere",
                })
                log_rows.append({
                    "target_location": loc,
                    "activity": act.get("name"),
                    "product": input_act.get("name"),
                    "original_location": input_act.get("location"),
                    "new_location": matched.get("location"),
                    "matched_name": best_name,
                    "similarity": round(score, 4),
                    "status": "changed",
                })

            new_db_data[localized_key] = new_act_data

    bd.Database(output_db_name).write(new_db_data)
    bd.Database(output_db_name).process()
    print(f"✅ Localized subset DB written to: {output_db_name}")

    pd.DataFrame(log_rows).to_excel(log_xlsx, index=False)
    print(f"📝 Log saved to: {log_xlsx}")

### LCIA Method Patching for Negative CO2

In [ ]:
def patch_negative_co2_in_methods(
    flow_name: str,
    flow_categories: tuple[str, str],
    keyword: str,
    log_xlsx: Path,
) -> None:
    """
    Add (flow_name, flow_categories) to any method that already contains at least one
    negative CF for a CO2-like flow (name contains keyword), but does not yet include this exact flow.
    Copies the negative CF value from existing CO2-like flow(s), requiring them to be consistent.
    
    Parameters
    ----------
    flow_name : str
        Name of the flow to add (e.g., "Carbon dioxide, in air")
    flow_categories : tuple[str, str]
        Categories of the flow (e.g., ("natural resource", "in air"))
    keyword : str
        Keyword to identify CO2-like flows (e.g., "carbon dioxide")
    log_xlsx : Path
        Path to save the log of added flows
    """
    # Locate the target flow in biosphere (not spatialized db)
    biosphere_db = bd.Database("ecoinvent-3.10.1-biosphere")

    flow_meta: Dict[tuple, dict] = {}
    target_flow_key = None

    for act in biosphere_db:
        key = act.key
        name = act.get("name")
        cats = tuple(act.get("categories", ()))
        flow_meta[key] = {"name": name, "name_lower": str(name).lower(), "categories": cats}
        if name == flow_name and cats == flow_categories:
            target_flow_key = key

    if target_flow_key is None:
        raise ValueError(f"Target flow '{flow_name}' with categories {flow_categories} not found in biosphere.")

    co2_like_keys = {k for k, m in flow_meta.items() if keyword in m["name_lower"]}

    def _to_float_amount(v):
        if isinstance(v, dict):
            return float(v.get("amount"))
        return float(v)

    log_records: List[dict] = []

    all_methods = list(bd.methods)
    for method_key in tqdm(all_methods, desc="Scanning methods", unit="method"):
        method = bd.Method(method_key)
        data_list = method.load()
        if not data_list:
            continue

        existing_keys = {fk for (fk, _) in data_list}
        if target_flow_key in existing_keys:
            continue

        candidates = []
        for fk, cf in data_list:
            if fk in co2_like_keys:
                try:
                    cf_num = _to_float_amount(cf)
                except Exception:
                    continue
                if cf_num < 0:
                    candidates.append((fk, cf_num))

        if not candidates:
            continue

        cf_values = {round(val, 12) for _, val in candidates}
        if len(cf_values) > 1:
            details = [(flow_meta[fk]["name"], flow_meta[fk]["categories"], val) for fk, val in candidates]
            raise ValueError(
                f"Method {method_key} has multiple different negative CFs for CO2-like flows: {cf_values}\nDetails: {details}"
            )

        source_flow_key, source_cf = candidates[0]

        new_data_list = list(data_list)
        new_data_list.append((target_flow_key, float(source_cf)))
        method.write(new_data_list)

        src_meta = flow_meta[source_flow_key]
        log_records.append({
            "method": str(method_key),
            "added_flow_name": flow_name,
            "added_flow_categories": " :: ".join(flow_categories),
            "source_flow_name": src_meta["name"],
            "source_flow_categories": " :: ".join(map(str, src_meta["categories"])),
            "copied_cf": float(source_cf),
        })

    if log_records:
        pd.DataFrame(log_records).to_excel(log_xlsx, index=False)
        print(f"✅ Added target flow to {len(log_records)} methods. Log saved to '{log_xlsx}'.")
    else:
        print("ℹ️ No methods required adding the target flow.")

## Execution Steps

### Step 1: Initialize Project

In [ ]:
_ensure_project(PROJECT)
print(f"✅ Working in project: {PROJECT}")

### Step 2: Import Methanol Foreground Database

In [ ]:
import_foreground_from_excel(
    excel_path=METHANOL_EXCEL,
    biosphere_db=BIOSPHERE_DB,
    technosphere_db=ECOINVENT_DB,
    extra_match_dbs=None,
    unlinked_csv=UNLINKED_METHANOL_CSV,
)

### Step 3: Post-Process Methanol Database

In [ ]:
# Remove self-referencing technosphere exchanges
removed = remove_self_referencing_technosphere_exchanges(METHANOL_FG_DB_NAME)
print(f"🧹 Removed {removed} self-referencing technosphere exchanges from '{METHANOL_FG_DB_NAME}'.")

# Replace hydrogen exchange in methanol production
repl, skipped = replace_technosphere_exchange_in_activity(
    db_name=METHANOL_FG_DB_NAME,
    target_activity_name="methanol production, from carbon dioxide and hydrogen",
    target_reference_product="methanol production, from carbon dioxide and hydrogen",
    old_input_names=[
        "consumption market for hydrogen, gaseous, medium pressure, merchant",
        "market for hydrogen, gaseous, medium pressure, merchant",
    ],
    old_input_reference_product="hydrogen, gaseous, medium pressure, merchant",
    new_input_name="hydrogen production, PEM electrolysis, green",
    new_input_reference_product="hydrogen production, PEM electrolysis, green",
)
print(f"🔁 Hydrogen exchange replacements: {repl} updated, {skipped} skipped.")

### Step 4: Localize Methanol Foreground Database

In [ ]:
# Load semantic matching model
model = SentenceTransformer(MODEL_NAME)

# Extract target locations from Regioinvent
methanol_location_source = METHANOL_BG_DBS_FOR_LOCALIZATION[METHANOL_EXTRACT_LOCATIONS_FROM_BG_INDEX]
methanol_target_locs = extract_target_locations_from_db(methanol_location_source)
print(f"🌍 Methanol target locations (n={len(methanol_target_locs)}): {methanol_target_locs[:10]}{'...' if len(methanol_target_locs)>10 else ''}")

# Localize methanol database
localize_foreground_database(
    original_fg_db_name=METHANOL_FG_DB_NAME,
    output_fg_db_name=METHANOL_LOCALIZED_DB,
    background_db_names=METHANOL_BG_DBS_FOR_LOCALIZATION,
    target_locations=methanol_target_locs,
    similarity_threshold=SIMILARITY_THRESHOLD,
    model=model,
    overwrite=OVERWRITE_DATABASES,
    mapping_log_xlsx=Path(f"exchange_mapping_log_{METHANOL_LOCALIZED_DB}.xlsx"),
)

### Step 5: Import Hydrogen Foreground Database

In [ ]:
import_foreground_from_excel(
    excel_path=HYDROGEN_EXCEL,
    biosphere_db=BIOSPHERE_DB,
    technosphere_db=ECOINVENT_DB,
    extra_match_dbs=[HYDROGEN_MATCH_EXTRA_FG_DB],
    unlinked_csv=UNLINKED_HYDROGEN_CSV,
)

### Step 6: Localize Hydrogen Foreground Database

In [ ]:
# Extract target locations from Regioinvent
hydrogen_location_source = HYDROGEN_BG_DBS_FOR_LOCALIZATION[HYDROGEN_EXTRACT_LOCATIONS_FROM_BG_INDEX]
hydrogen_target_locs = extract_target_locations_from_db(hydrogen_location_source)
print(f"🌍 Hydrogen target locations (n={len(hydrogen_target_locs)}): {hydrogen_target_locs[:10]}{'...' if len(hydrogen_target_locs)>10 else ''}")

# Localize hydrogen database
localize_foreground_database(
    original_fg_db_name=HYDROGEN_FG_DB_NAME,
    output_fg_db_name=HYDROGEN_LOCALIZED_DB,
    background_db_names=HYDROGEN_BG_DBS_FOR_LOCALIZATION,
    target_locations=hydrogen_target_locs,
    similarity_threshold=SIMILARITY_THRESHOLD,
    model=model,
    overwrite=OVERWRITE_DATABASES,
    mapping_log_xlsx=Path(f"exchange_mapping_log_{HYDROGEN_LOCALIZED_DB}.xlsx"),
)

### Step 7 (Optional): Build Localized Ecoinvent Subset Database

In [ ]:
if BUILD_LOCALIZED_ECOINVENT_SUBSET:
    subset_loc_source = ECOINVENT_SUBSET_BG_DBS[ECOINVENT_SUBSET_EXTRACT_LOCATIONS_FROM_BG_INDEX]
    subset_target_locs = extract_target_locations_from_db(subset_loc_source)
    print(f"🌍 Subset target locations (n={len(subset_target_locs)}).")

    localize_ecoinvent_subset(
        source_db_name=ECOINVENT_SUBSET_SOURCE_DB,
        output_db_name=ECOINVENT_SUBSET_OUTPUT_DB,
        background_db_names=ECOINVENT_SUBSET_BG_DBS,
        target_locations=subset_target_locs,
        targets=ECOINVENT_SUBSET_TARGET_ACTS,
        model=model,
        similarity_threshold=SIMILARITY_THRESHOLD,
        overwrite=OVERWRITE_DATABASES,
        log_xlsx=Path(f"exchange_mapping_log_{ECOINVENT_SUBSET_OUTPUT_DB}.xlsx"),
    )
else:
    print("⏭️ Skipping localized ecoinvent subset creation.")

### Step 8 (Optional): Patch LCIA Methods for Negative CO2

In [ ]:
if PATCH_NEGATIVE_CO2:
    patch_negative_co2_in_methods(
        flow_name=CO2_FLOW_NAME,
        flow_categories=CO2_FLOW_CATEGORIES,
        keyword=CO2_KEYWORD,
        log_xlsx=CO2_PATCH_LOG_XLSX,
    )
else:
    print("⏭️ Skipping CO2 method patching.")


### Step 9: Completion Summary

In [ ]:
print("\n" + "="*60)
print("✅ PROJECT SETUP COMPLETE")
print("="*60)
print(f"\nDatabases created:")
print(f"  • {METHANOL_LOCALIZED_DB}")
print(f"  • {HYDROGEN_LOCALIZED_DB}")
if BUILD_LOCALIZED_ECOINVENT_SUBSET:
    print(f"  • {ECOINVENT_SUBSET_OUTPUT_DB}")
print(f"\nAll databases are ready for use in project: {PROJECT}")